# EXP-RS-07 — Sheaves-LBD v0.1

**Purpose:** Validate the cellular-sheaf bridge detector on (a) the Hopfield/Spin-glass toy fixture
and (b) the pre-2015 Feynman 10-pair arXiv corpus. The sheaf Laplacian L_F = δᵀδ is assembled
over the Louvain community graph; inter-community edges with high frustration Φ(e) are the
predicted LBD bridges.

| Test | What it checks |
|------|----------------|
| T1 synthetic | Ground-truth bridge in top-3 on 2-community toy (algorithm correctness) |
| T2 10-pair | precision@10 ≥ 0.4 on Feynman corpus (ceiling 0.5 — 5 evaluable pairs) |
| T4 ablation | Top-5 bridge ablation ratios all > 3 (multi-causal verification) |
| SC1 H⁰(F) | dim(ker L_F) ≤ 5 (warn if disconnected sheaf) |
| SC2 symmetry | ‖L−Lᵀ‖_F/‖L‖_F < 1e-6 (abort on block-assembly bug) |
| SC3 PSD | min eigenvalue ≥ −1e-8 (abort if not positive semi-definite) |
| SC4 spectral gap | λ₂₁/λ₂₀ ≥ 1.1 (flag near-degenerate near-sections) |
| SC5 Jaccard | J(top-10 sheaf, top-10 cosine) ≤ 0.8 (flag if re-implementing baseline) |

**Phase-43 gate:** T1 PASS AND T2 precision@10 ≥ 0.4 AND T4 pass-rate ≥ 50% → AUTHORIZE.

## Algorithm sketch

1. **Load** — parse the community-graph JSON export (`communities[].tfidf_vec` → vertex stalks).
2. **Stalks** — top-100 c-TF-IDF terms per community; no client-side re-aggregation.
3. **Restriction maps** — for each inter-community edge (c₁, c₂): shared = basis(c₁) ∩ basis(c₂);
   build two sparse selection matrices F_{c₁}, F_{c₂} of shape (d_e × 100).
4. **Laplacian** — sparse block-COO assembly: diagonal blocks += Fᵀ F; off-diagonal blocks −= Fᵀ F'.
5. **Eigensolve** — dense eigh for N ≤ 500 (toy); sparse eigsh(sigma=1e-8) for large corpus.
6. **Frustration** — φ_i(e) = ‖F_{c₁} x_{c₁} − F_{c₂} x_{c₂}‖² / (‖x_{c₁}‖² + ‖x_{c₂}‖²);
   Φ(e) = Σ_i φ_i(e) / λ_i.
7. **T4 ablation** — zero dominant vertex blocks from near-section i*; ratio = λ_ablated / λ_{i*}.

In [ ]:
import subprocess, sys
result = subprocess.run(
    [sys.executable, 'prototypes/sheaf_lbd_v01.py'],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)

## SC1 note — large H⁰(F) on the toy is expected

The toy has 2 communities each with a 12-term stalk; only 2 terms are shared (spin, ising).
The kernel of L_F includes all un-constrained coordinates (10 per community) plus 2 consistent
shared sections → H⁰ = 22. This is a design property of the small toy, not an algorithm failure.
The full Feynman corpus has many more inter-community shared terms, so H⁰ should be ≤ 5.

## Results summary

See `prototypes/SHEAF_V01_RESULTS.md` for the full test table and Phase-43 verdict.

| Test | Result |
|------|--------|
| T1 toy fixture | HOLD — run notebook to populate |
| T2 precision@10 | HOLD — Feynman corpus pending |
| T4 ablation | HOLD — Feynman corpus pending |
| **Phase-43 decision** | **HOLD** |